Step 5. Correct zero distribution

The βVAE model interpolation is smooth in the latent space and decreases dropout rate, which may distort the reliable biological structure of the simulated data. To better simulate the dropouts in real scRNA-seq datasets, the followig script estimates the optimal dropout ratio and truncate the expression distribution.

In [ ]:
# Inspect the dropout rate of the input datasets.
X = adata.X
zeros_per_cell = (X == 0).sum(axis=1)/len(adata.var.index)
adata.X.sum(axis = 1)
adata.obs['n_zero_genes'] = zeros_per_cell
sns.violinplot(
    x="time",
    y="n_zero_genes",
    data=adata.obs,
    palette="Set1",
    inner="quartile", 
    cut=0,
    saturation=1
)
plt.ylim(0.8, 1)

In [ ]:
# run this for sparse matrix
non_zero_counts = adata.X.indptr[1:] - adata.X.indptr[:-1]
adata.obs['n_zero_genes'] = 1 - (non_zero_counts / len(adata.var.index))
sns.violinplot(
    x="time",
    y="n_zero_genes",
    data=adata.obs,
    palette="Set1",
    inner="quartile", 
    cut=0,
    saturation=1
)

For each input matrix, the following script computes its global dropout rate (fraction of zero entries). These n (number of input datasets) dropout rates are sorted in ascending order and assigned ranks x (x=1, 2, …, n).

In [ ]:
q_s = np.percentile(adata[adata.obs.s_e == 'start'].obs['n_zero_genes'], 50)
q_e = np.percentile(adata[adata.obs.s_e == 'end'].obs['n_zero_genes'], 50)

In [ ]:
q_values = [q_s, q_e]
x = np.array(list(range(len(q_values))))
y = np.array(sorted(q_values))

If the number of input timepoints is larger than 2, use the following steps to measure the optimal range of dropout ratios. Specifically, we fit a simple linear trend describing how the dropout rate changes with rank. Using this fitted trend, it estimates the dropout rate expected at the target timepoint, chosen as the average rank among all input datasets. To account for uncertainty, the prediction interval around this estimated dropout rate iscalculated. This interval gives a plausible lower and upper bound for the dropout rate at the target timepoint. Finally, the dropout-rate bounds were converted back into the corresponding X value range.

In [ ]:
x_mean = np.mean(x)
y_mean = np.mean(y)
b = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean)**2)
a = y_mean - b * x_mean
y_pred = a + b * x_mean
y_pred_train = a + b * x
residuals = y - y_pred_train
sse = np.sum(residuals**2)
sigma_hat = np.sqrt(sse / (len(q_values) - 2))
se_pred = sigma_hat * np.sqrt(1 + 1/5 + (len(q_values) - 2 - x_mean)**2 / np.sum((x - x_mean)**2))

In [ ]:
a_vals = np.linspace(0.001, 0.95, 100)

t_crit_vals = stats.t.ppf(1 - a_vals/2, df=len(q_values)-2)
lower = y_pred - t_crit_vals * se_pred
upper = y_pred + t_crit_vals * se_pred

plt.figure(figsize=(7, 5))

upper[upper > 1] = 1.0
plt.plot(a_vals, lower, label='Lower bound', color='blue')
plt.plot(a_vals, upper, label='Upper bound', color='red')
a_mark = 0.5 # if multiple time points are involved, a_mark can be set larger with more strict statistical test
mask = a_vals >= a_mark
plt.fill_between(a_vals, lower, upper, where=mask, alpha=0.2, color='gray', interpolate=True, label='Prediction interval')

plt.axhline(y=y_pred, color='green', linestyle='--', linewidth=1.5, label=f'pred = {y_pred:.3f}')
plt.text(0.5, y_pred + 0.002, f'pred = {y_pred:.3f}', ha='right', va='bottom', fontsize=10, color='green')

t_mark = stats.t.ppf(1 - a_mark/2, df=3)
low_mark = y_pred - t_mark * se_pred
up_mark = y_pred + t_mark * se_pred

plt.scatter(a_mark, low_mark, s=50, color='blue', edgecolors='k', zorder=5)
plt.scatter(a_mark, up_mark, s=50, color='red', edgecolors='k', zorder=5)
plt.legend(loc='lower right', fontsize=9, framealpha=0.8)
plt.xlabel('Significance level α')
plt.ylabel('Dropout rate (%)')
plt.title('Prediction interval of dropout rate vs. α')
plt.grid(True, alpha=0.3)

plt.text(a_mark, low_mark - 0.012, f'({a_mark:.2f}, {low_mark:.3f})', 
         ha='center', fontsize=9, color='blue')
plt.text(a_mark, up_mark + 0.012, f'({a_mark:.2f}, {up_mark:.3f})', 
         ha='center', fontsize=9, color='red')
plt.show()

In [ ]:
cell_orders = preprocess_cell_orders(adata_interpolated)
X_candidates = np.arange(10000, 20001, 500)
dropout_medians = []

for X in X_candidates:
    med = median_dropout_for_X(cell_orders, X)
    dropout_medians.append(med)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(X_candidates, dropout_medians, 'o-', color='orange', label='Median dropout rate', markersize=3)

plt.axhline(y=y_pred, color='green', linestyle='--', label=f'Predicted median = {y_pred:.3f}', alpha=0.9)
plt.axhline(y=up_mark, color='red', linestyle='-', label=f'Upper bound = {up_mark:.3f}', alpha=0.5)
plt.axhline(y=low_mark, color='blue', linestyle='-', label=f'Lower bound = {low_mark:.3f}', alpha=0.5)

f_interp = interp1d(dropout_medians, X_candidates, kind='linear', bounds_error=False, fill_value=(X_candidates[0], X_candidates[-1]))
X_at_lower = f_interp(low_mark)
X_at_upper = f_interp(up_mark)
X_at_pred = f_interp(y_pred)

if not np.isnan(X_at_lower):
    plt.scatter(X_at_lower, low_mark, color='blue', zorder=5, marker='s', alpha=0.75)
    plt.text(X_at_lower, low_mark-0.01, f'X={X_at_lower:.0f}', ha='center', fontsize=8)
if not np.isnan(X_at_upper):
    plt.scatter(X_at_upper, up_mark, color='red', zorder=5, marker='s', alpha=0.75)
    plt.text(X_at_upper, up_mark+0.01, f'X={X_at_upper:.0f}', ha='center', fontsize=8)
if not np.isnan(X_at_pred):
    plt.scatter(X_at_pred, y_pred, color='green', zorder=5, marker='s', alpha=0.75)
    plt.text(X_at_pred, y_pred-0.015, f'X={X_at_pred:.0f}', ha='center', fontsize=8, color='green')

if not np.isnan(X_at_lower) and not np.isnan(X_at_upper):
    X_min = min(X_at_lower, X_at_upper)
    X_max = max(X_at_lower, X_at_upper)
    plt.axvspan(X_min, X_max, alpha=0.2, color='gray', label=f'Reasonable X: [{X_min:.0f}, {X_max:.0f}]')

plt.xlabel('Initial scale total UMI (X)')
plt.ylabel('Median dropout rate')
plt.title('Dropout rate vs. initial scale X')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

If only two timepoints were used for simulation, use the following simpler steps to measure the optimal range of dropout ratio

In [ ]:
x_mean = np.mean(x)
y_mean = np.mean(y)
y_pred = y_mean

In [ ]:
a_vals = np.linspace(0.001, 0.95, 100)
data_range = max(q_s, q_e) - min(q_s, q_e)
width = data_range * (1 - a_vals)

lower = y_pred - width / 2
upper = y_pred + width / 2

plt.figure(figsize=(7, 5))

upper[upper > 1] = 1.0
plt.plot(a_vals, lower, label='Lower bound', color='blue')
plt.plot(a_vals, upper, label='Upper bound', color='red')
a_mark = 0.2 # if only 2 time points are involved, a_mark should be set lower with less strict statistical test
mask = a_vals >= a_mark
plt.fill_between(a_vals, lower, upper, where=mask, alpha=0.2, color='gray', interpolate=True, label='Prediction interval')

plt.axhline(y=y_pred, color='green', linestyle='--', linewidth=1.5, label=f'pred = {y_pred:.3f}')
plt.text(0.5, y_pred + 0.002, f'pred = {y_pred:.3f}', ha='right', va='bottom', fontsize=10, color='green')

low_mark = y_pred - (data_range * (1 - a_mark)) / 2
up_mark = y_pred + (data_range * (1 - a_mark)) / 2
low_mark = np.clip(low_mark, min(q_s, q_e), max(q_s, q_e))
up_mark = np.clip(up_mark, min(q_s, q_e), max(q_s, q_e))

plt.scatter(a_mark, low_mark, s=50, color='blue', edgecolors='k', zorder=5)
plt.scatter(a_mark, up_mark, s=50, color='red', edgecolors='k', zorder=5)
plt.legend(loc='lower right', fontsize=9, framealpha=0.8)
plt.xlabel('Significance level α')
plt.ylabel('Dropout rate (%)')
plt.title('Prediction interval of dropout rate vs. α')
plt.grid(True, alpha=0.3)

plt.text(a_mark, low_mark - 0.005, f'({a_mark:.2f}, {low_mark:.3f})', 
         ha='center', fontsize=9, color='blue')
plt.text(a_mark, up_mark + 0.005, f'({a_mark:.2f}, {up_mark:.3f})', 
         ha='center', fontsize=9, color='red')
plt.show()

In [ ]:
cell_orders = preprocess_cell_orders(adata_interpolated)
X_candidates = np.arange(40001, 110001, 2000)
dropout_medians = []

for X in X_candidates:
    med = median_dropout_for_X(cell_orders, X)
    dropout_medians.append(med)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(S_candidates, dropout_medians, 'o-', color='orange', label='Median dropout rate', markersize=3)

plt.axhline(y=y_pred, color='green', linestyle='--', label=f'Predicted median = {y_pred:.3f}', alpha=0.9)
plt.axhline(y=up_mark, color='red', linestyle='-', label=f'Upper bound = {up_mark:.3f}', alpha=0.5)
plt.axhline(y=low_mark, color='blue', linestyle='-', label=f'Lower bound = {low_mark:.3f}', alpha=0.5)

f_interp = interp1d(dropout_medians, X_candidates, kind='linear', bounds_error=False, fill_value=(X_candidates[0], X_candidates[-1]))
X_at_lower = f_interp(low_mark)
X_at_upper = f_interp(up_mark)
X_at_pred = f_interp(y_pred)

if not np.isnan(X_at_lower):
    plt.scatter(X_at_lower, low_mark, color='blue', zorder=5, marker='s', alpha=0.75)
    plt.text(X_at_lower, low_mark-0.005, f'X={X_at_lower:.0f}', ha='center', fontsize=8, color='blue')
if not np.isnan(X_at_upper):
    plt.scatter(X_at_upper, up_mark, color='red', zorder=5, marker='s', alpha=0.75)
    plt.text(X_at_upper, up_mark-0.005, f'X={X_at_upper:.0f}', ha='center', fontsize=8, color='red')
if not np.isnan(X_at_pred):
    plt.scatter(X_at_pred, y_pred, color='green', zorder=5, marker='s', alpha=0.75)
    plt.text(X_at_pred, y_pred-0.005, f'X={X_at_pred:.0f}', ha='center', fontsize=8, color='green')

if not np.isnan(X_at_lower) and not np.isnan(X_at_upper):
    X_min = min(X_at_lower, X_at_upper)
    X_max = max(X_at_lower, X_at_upper)
    plt.axvspan(X_min, X_max, alpha=0.2, color='gray', label=f'Reasonable X: [{X_min:.0f}, {X_max:.0f}]')

plt.xlabel('Initial scale total UMI (X)')
plt.ylabel('Median dropout rate')
plt.title('Dropout rate vs. initial scale X')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

After determining the best dropout ratio for the simulated data, run the following script to perform the normalization.

Note that this script is intended for single execution.

To regenerate adata_interpolated after target_sum changes, execute the reset script (labeled 'generate or reset adata_interpolated') in the step 4 notebook before running this cell again.

In [ ]:
sc.pp.normalize_total(adata_interpolated, target_sum=5.5e4) # target_sum is the calculated X
adata_interpolated.X = reduce_total_expression(adata_interpolated.X, target_sum=1e4)

X = adata_interpolated.X
zeros_per_cell = (X == 0).sum(axis=1)/len(adata_interpolated.var.index)
adata_interpolated.X.sum(axis = 1)
adata_interpolated.obs['n_zero_genes'] = zeros_per_cell

sns.violinplot(
    x="time",
    y="n_zero_genes",
    data=adata_interpolated.obs,
    palette="Set1",
    inner="quartile",
    cut=2,
    saturation=1
)
plt.ylim(0.8, 1)

Save the final output

In [ ]:
adata_interpolated.write_h5ad()